### Merge first-order features with the target label
Combining the pre-outcome features (built from each customer's first order only) 
with the second-purchase label from the EDA step. Only the valid cohort — customers 
with enough observation time — gets kept.

In [1]:
import pandas as pd
import numpy as np

first_order_features = pd.read_csv("../data/processed/first_order_features.csv")
labeled_customers = pd.read_csv("../data/processed/customer_features_labeled.csv")

# Keep only the target-relevant columns from the labeled table —
# everything else there was computed across ALL orders and would leak
target_cols = labeled_customers[["customer_unique_id", "made_second_purchase"]]

# Inner join: only customers present in both (valid cohort AND has first-order features)
model_data = first_order_features.merge(target_cols, on="customer_unique_id", how="inner")

print(f"Merged dataset: {model_data.shape[0]:,} rows, {model_data.shape[1]} columns")
print(f"Target balance: {model_data['made_second_purchase'].mean():.2%} positive")
model_data.head()

Merged dataset: 55,907 rows, 15 columns
Target balance: 3.97% positive


,customer_unique_id,customer_state,first_purchase_date,n_items,n_distinct_products,items_total_price,total_freight,product_category_name,payment_total,max_installments,payment_type,delivery_days,delivery_delay_days,review_score,made_second_purchase
0,df9d2585c3e7b4f8904b1933e5ad2c7c,SP,2017-07-17 15:30:54,4,4,136.00,31.12,relogios_presentes,167.12,8.0,credit_card,2.0,-9.0,4.0,0
1,5bcfa017a23c7190fcdfc9fd753bd174,TO,2017-10-16 12:29:49,2,2,108.00,47.70,moveis_decoracao,155.70,1.0,boleto,12.0,-10.0,5.0,0
2,34520a1612de8199cc5a1e873db564fa,RJ,2017-07-18 18:22:50,1,1,49.90,14.10,fashion_bolsas_e_acessorios,64.00,1.0,boleto,14.0,-20.0,5.0,0
3,c8f25042cc8c47c3dee2541f1ca92b93,SP,2017-10-16 13:40:00,1,1,12.99,9.34,moveis_decoracao,22.33,2.0,credit_card,2.0,-8.0,5.0,0
4,64870b178a9fb3eb16dd905d8668274c,SC,2017-04-15 23:46:11,1,1,97.00,14.85,ferramentas_jardim,111.85,2.0,credit_card,25.0,0.0,5.0,0


In [2]:
model_data.isnull().sum()

customer_unique_id          0
customer_state              0
first_purchase_date         0
n_items                     0
n_distinct_products         0
items_total_price           0
total_freight               0
product_category_name    1021
payment_total               1
max_installments            1
payment_type                1
delivery_days               2
delivery_delay_days         2
review_score              418
made_second_purchase        0
dtype: int64

### Check for missing values
Payment, review, and delivery fields can be null — reviews may not exist yet, and 
some orders may be missing delivery timestamps. Deciding how to handle each before 
modeling.

In [3]:
missing_summary = model_data.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_summary / len(model_data) * 100).round(2)
pd.DataFrame({"missing_count": missing_summary, "missing_pct": missing_pct})

,missing_count,missing_pct
product_category_name,1021,1.83
review_score,418,0.75
delivery_days,2,0.00
delivery_delay_days,2,0.00
payment_total,1,0.00
max_installments,1,0.00
payment_type,1,0.00
total_freight,0,0.00
items_total_price,0,0.00
n_distinct_products,0,0.00


### Handling missing values
Each column's gap has a different cause, so each gets its own fix rather than 
one blanket rule. Category and review gaps are meaningful (no data left, not 
necessarily bad data) and get flagged/filled. The handful of rows missing 
payment or delivery info are just dropped too few to matter.


In [4]:
# Drop the tiny number of rows with missing payment/delivery info —
# not enough rows to justify imputation, and they may indicate
# a broken order record anyway
model_data = model_data.dropna(
    subset=["delivery_days", "delivery_delay_days", "payment_total",
            "max_installments", "payment_type"]
)

# Missing category likely means the product had no listed category in
# the source data — treat "unknown" as its own valid category rather
# than dropping potentially useful rows
model_data["product_category_name"] = model_data["product_category_name"].fillna("unknown")

# Missing review means no review was left — that's different from a bad
# review, so we flag it explicitly and fill the score with the median
# (keeps the average review score undistorted)
model_data["has_review"] = model_data["review_score"].notnull().astype(int)
model_data["review_score"] = model_data["review_score"].fillna(model_data["review_score"].median())

print(f"Remaining rows after cleanup: {len(model_data):,}")
print(f"Remaining nulls:\n{model_data.isnull().sum().sum()}")

Remaining rows after cleanup: 55,904
Remaining nulls:
0


### Encoding categorical columns
Different cardinality gets different treatment. Payment type and state are 
low/medium cardinality, so one-hot encoding is fine. Product category has 70+ 
values — one-hot would blow up the feature space, so it gets frequency encoded 
instead.

In [6]:
# One-hot encode low/medium cardinality columns
model_data = pd.get_dummies(
    model_data,
    columns=["payment_type", "customer_state"],
    prefix=["payment", "state"],
    drop_first=True  # avoid multicollinearity — drop one reference category per column
)

# Frequency encode product_category_name — replace each category with
# how often it appears in the data, avoids a 70+ column explosion
category_freq = model_data["product_category_name"].value_counts(normalize=True)
model_data["category_frequency"] = model_data["product_category_name"].map(category_freq)
model_data = model_data.drop(columns=["product_category_name"])

print(f"Final shape: {model_data.shape[0]:,} rows, {model_data.shape[1]} columns")
model_data.dtypes.value_counts()

Final shape: 55,904 rows, 43 columns


bool       29
float64     8
int64       4
str         2
Name: count, dtype: int64

### Save the model-ready table
This is the final table feature engineering produces — everything downstream 
(train/test split, baseline model) reads from here.

In [7]:
output_path = "../data/processed/model_ready.csv"
model_data.to_csv(output_path, index=False)
print(f"Saved model-ready table to {output_path}")
print(f"Shape: {model_data.shape}")

Saved model-ready table to ../data/processed/model_ready.csv
Shape: (55904, 43)
